### Agent的高级用法-ToolStrategy

#### 1. ToolStrategy 的多种结构化输出方式: schema 参数

##### 1.1 Pydantic 类型

In [12]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, SystemMessage
import os
from langchain.agents import create_agent
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy, AutoStrategy
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain_tavily import TavilySearch
from dotenv import load_dotenv
from rich import print as rprint
from pydantic import BaseModel, Field

load_dotenv(override=True)
ZHIPUAI_API_KEY = os.getenv("ZHIPUAI_API_KEY")
ZHIPUAI_API_BASE_URL = os.getenv("ZHIPUAI_API_BASE_URL")

# 1. 模型初始化
model = init_chat_model(
    model='glm-5.3',
    model_provider="openai",
    api_key=ZHIPUAI_API_KEY,
    base_url=ZHIPUAI_API_BASE_URL
)

举例1: 

In [13]:
# 2. 使用 Pydantic 结构化方式定义
class ContractInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户的名称")
    email: str = Field(description="用户的邮箱")
    phone: str = Field(description="用户的电话")

agent = create_agent(
    model=model,
    response_format=AutoStrategy(ContractInfo)
)

response = agent.invoke({
    "messages": [
        HumanMessage(content="从这段话中抽取结构化信息: 小明的邮箱是shkstart@atguigu.com, 电话是: 123123123123")
    ]
})

rprint(response)

{
    'messages': [
        HumanMessage(
            content='从这段话中抽取结构化信息: 小明的邮箱是shkstart@atguigu.com, 电话是: 123123123123',
            additional_kwargs={},
            response_metadata={},
            id='545872ae-d906-4895-94cb-c20120c87d56'
        ),
        AIMessage(
            content='我从这段话中抽取到以下结构化信息,并调用工具记录:',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 128,
                    'prompt_tokens': 225,
                    'total_tokens': 353,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 78,
                        'rejected_prediction_tokens': None
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'glm-5.3',
                'system_fingerprint': None,
                'id': '20261113155732a6dbefebf4154aac',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a1d3e9-6acd-7511-a3c5-0599ed3ec0d1-0',
            tool_calls=[
                {
                    'name': 'ContractInfo',
                    'args': {'email': 'shkstart@atguigu.com', 'name': '小明', 'phone': '123123123123'},
                    'id': 'call_-7080015989865184987',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 225,
                'output_tokens': 128,
                'total_tokens': 353,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {'reasoning': 78}
            }
        ),
        ToolMessage(
            content="Returning structured response: name='小明' email='shkstart@atguigu.com' phone='123123123123'",
            name='ContractInfo',
            id='b0daff64-1066-42ec-af9d-4aad05f35ba3',
            tool_call_id='call_-7080015989865184987'
        )
    ],
    'structured_response': ContractInfo(name='小明', email='shkstart@atguigu.com', phone='123123123123')
}

举例2: 添加工具的调用

In [19]:
from typing import Literal

# 定义工具
@tool(parse_docstring=True)
def search_customer_database(query: str) -> str:
    """
    在客户数据库中搜索信息

    Args:
        query (str): 客户查询字符串, 例如 "张三" 或 "李四"

    Returns:
        str: 客户记录字符串, 包含客户姓名, 等级, 最近购买日期和累计消费
    """
    # 模拟数据库查询结果
    if "张三" in query.lower():
        return "客户记录: 张三, VIP 客户, 最近购买日期: 2026-01-15, 累计消费: $15,000"
    elif "李四" in query.lower():
        return "客户记录: 李四, 普通客户, 最近购买日期: 2025-12-20, 累计消费: $3,200"
    else:
        return f"关于客户{query}, 无记录"

@tool(parse_docstring=True)
def send_email(customer: str) -> str:
    """
    发送感谢邮件

    Args:
        customer (str): 客户名称, 例如 "张三" 或 "李四"

    Returns:
        str: 确认消息, 包含已发送的客户名称
    """
    return f"已向 {customer} 发送感谢邮箱"

# 定义 Pydantic Schema
class CustomerAnalysis(BaseModel):
    """客户分析报告"""
    customer_name: str = Field(None, description="客户姓名")
    customer_tier: Literal["潜在客户", "普通客户", "VIP客户", "流式风险"] = Field("潜在客户", description="客户等级, 只能是潜在客户, 普通客户, VIP客户或流式风险")
    spending_level: Literal["低", "中", "高"] = Field(None, description="消费水平")
    send_email: bool = Field(False, description="是否已发送感谢邮件")

# 创建智能体
agent = create_agent(
    model=model,
    system_prompt=SystemMessage(content=""
                                "请分析指定客户的情况:"
                                "1. 先搜索客户数据库了解最新情况"
                                "2. 如果是 VIP 客户, 则发送感谢邮件"
                                "3. 基于搜索结果生成结构化分析报告"
                                "4. 如果用户提问与客户记录无关或找不到客户信息, 则返回空对象, 不发送感谢邮件"
                                ),
    tools=[search_customer_database, send_email],
    response_format=ToolStrategy(schema=CustomerAnalysis)
)

# 执行分析
result = agent.invoke({
    # "messages": [{"role": "user", "content": "请分析客户张三"}]
    "messages": [{"role": "user", "content": "请分析客户李四"}]
    # "messages": [{"role": "user", "content": "请分析客户王五"}]
    # "messages": [{"role": "user", "content": "今天天气如何"}]
})

# 处理结果
rprint(result)

{
    'messages': [
        HumanMessage(
            content='请分析客户李四',
            additional_kwargs={},
            response_metadata={},
            id='7d347c39-0e81-4f54-9dc7-9ef5a54c8a0d'
        ),
        AIMessage(
            content='我来帮您分析客户李四的情况。首先搜索客户数据库了解最新情况。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 159,
                    'prompt_tokens': 467,
                    'total_tokens': 626,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 128,
                        'rejected_prediction_tokens': None
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'glm-5.3',
                'system_fingerprint': None,
                'id': '20261113160612abeaa63ed5aa44e2',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a1d3f1-5adf-75e1-9053-636340be7d88-0',
            tool_calls=[
                {
                    'name': 'search_customer_database',
                    'args': {'query': '李四'},
                    'id': 'call_-7080016745779425799',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 467,
                'output_tokens': 159,
                'total_tokens': 626,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {'reasoning': 128}
            }
        ),
        ToolMessage(
            content='客户记录: 李四, 普通客户, 最近购买日期: 2025-12-20, 累计消费: $3,200',
            name='search_customer_database',
            id='226cae0c-4ccd-422c-8f1e-313ecc246a34',
            tool_call_id='call_-7080016745779425799'
        ),
        AIMessage(
            content='已找到客户李四的信息。由于李四是**普通客户**（非VIP客户），根据分析流程，无需发送感谢邮件。现
在基于搜索结果生成结构化分析报告：',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 252,
                    'prompt_tokens': 536,
                    'total_tokens': 788,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 173,
                        'rejected_prediction_tokens': None
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'glm-5.3',
                'system_fingerprint': None,
                'id': '20261113160617e83369aa1f154c9d',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a1d3f1-6d14-7eb2-adb5-b5e333b9ca7b-0',
            tool_calls=[
                {
                    'name': 'CustomerAnalysis',
                    'args': {
                        'customer_name': '李四',
                        'customer_tier': '普通客户',
                        'send_email': False,
                        'spending_level': '中'
                    },
                    'id': 'call_-7080015886785963846',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 536,
                'output_tokens': 252,
                'total_tokens': 788,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {'reasoning': 173}
            }
        ),
        ToolMessage(

### 1.2 TypedDict 类型

举例1

In [20]:
from typing import TypedDict, Annotated

# 2. 使用 Pydantic 结构化方式定义
class ContractInfo(TypedDict):
    """用户的联系方式"""
    name: str = Annotated[str, ..., "用户的名称"]
    email: str = Annotated[str, ..., "用户的邮箱"]
    phone: str = Annotated[str, ..., "用户的电话"]

agent = create_agent(
    model=model,
    response_format=AutoStrategy(ContractInfo)
)

response = agent.invoke({
    "messages": [
        HumanMessage(content="从这段话中抽取结构化信息: 小明的邮箱是shkstart@atguigu.com, 电话是: 123123123123")
    ]
})

rprint(response)

{
    'messages': [
        HumanMessage(
            content='从这段话中抽取结构化信息: 小明的邮箱是shkstart@atguigu.com, 电话是: 123123123123',
            additional_kwargs={},
            response_metadata={},
            id='56de22c0-c014-4995-b5e3-f896119835ca'
        ),
        AIMessage(
            content='我从这段话中抽取到以下结构化信息:\n\n- **姓名**: 小明\n- **邮箱**: shkstart@atguigu.com\n- 
**电话**: 123123123123\n\n现在我将这些信息提交:',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 215,
                    'prompt_tokens': 204,
                    'total_tokens': 419,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 134,
                        'rejected_prediction_tokens': None
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'glm-5.3',
                'system_fingerprint': None,
                'id': '202611161054350b48b5d2174b4452',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a1e247-220f-72f3-a9a2-aaa8517d5695-0',
            tool_calls=[
                {
                    'name': 'ContractInfo',
                    'args': {'email': 'shkstart@atguigu.com', 'name': '小明', 'phone': '123123123123'},
                    'id': 'call_-7080003311121725659',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 204,
                'output_tokens': 215,
                'total_tokens': 419,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {'reasoning': 134}
            }
        ),
        ToolMessage(
            content="Returning structured response: {'name': '小明', 'email': 'shkstart@atguigu.com', 'phone': 
'123123123123'}",
            name='ContractInfo',
            id='e7a1bc85-ff13-4a53-91a5-437b87d710a6',
            tool_call_id='call_-7080003311121725659'
        )
    ],
    'structured_response': {'name': '小明', 'email': 'shkstart@atguigu.com', 'phone': '123123123123'}
}

举例2

In [21]:
from typing import Literal, Optional

# 定义工具
@tool(parse_docstring=True)
def search_customer_database(query: str) -> str:
    """
    在客户数据库中搜索信息

    Args:
        query (str): 客户查询字符串, 例如 "张三" 或 "李四"

    Returns:
        str: 客户记录字符串, 包含客户姓名, 等级, 最近购买日期和累计消费
    """
    # 模拟数据库查询结果
    if "张三" in query.lower():
        return "客户记录: 张三, VIP 客户, 最近购买日期: 2026-01-15, 累计消费: $15,000"
    elif "李四" in query.lower():
        return "客户记录: 李四, 普通客户, 最近购买日期: 2025-12-20, 累计消费: $3,200"
    else:
        return f"关于客户{query}, 无记录"

@tool(parse_docstring=True)
def send_email(customer: str) -> str:
    """
    发送感谢邮件

    Args:
        customer (str): 客户名称, 例如 "张三" 或 "李四"

    Returns:
        str: 确认消息, 包含已发送的客户名称
    """
    return f"已向 {customer} 发送感谢邮箱"

# 定义 Pydantic Schema
class CustomerAnalysis(TypedDict):
    """客户分析报告"""
    customer_name: Annotated[Optional[str], ..., "客户姓名"]
    customer_tier: Annotated[Literal["潜在客户", "普通客户", "VIP客户", "流式风险"], ..., "客户等级, 只能是潜在客户, 普通客户, VIP客户或流式风险"]
    spending_level: Annotated[Literal["低", "中", "高"], ..., "消费水平"]
    send_email: Annotated[bool, ..., "是否已发送感谢邮件"]

# 创建智能体
agent = create_agent(
    model=model,
    system_prompt=SystemMessage(content=""
                                "请分析指定客户的情况:"
                                "1. 先搜索客户数据库了解最新情况"
                                "2. 如果是 VIP 客户, 则发送感谢邮件"
                                "3. 基于搜索结果生成结构化分析报告"
                                "4. 如果用户提问与客户记录无关或找不到客户信息, 则返回空对象, 不发送感谢邮件"
                                ),
    tools=[search_customer_database, send_email],
    response_format=ToolStrategy(schema=CustomerAnalysis)
)

# 执行分析
result = agent.invoke({
    # "messages": [{"role": "user", "content": "请分析客户张三"}]
    "messages": [{"role": "user", "content": "请分析客户李四"}]
    # "messages": [{"role": "user", "content": "请分析客户王五"}]
    # "messages": [{"role": "user", "content": "今天天气如何"}]
})

# 处理结果
rprint(result)

{
    'messages': [
        HumanMessage(
            content='请分析客户李四',
            additional_kwargs={},
            response_metadata={},
            id='0cc9790d-3b88-443e-b5a6-aa5b9f197723'
        ),
        AIMessage(
            content='我来帮您分析客户李四的情况。首先搜索客户数据库获取最新信息:',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 152,
                    'prompt_tokens': 437,
                    'total_tokens': 589,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 121,
                        'rejected_prediction_tokens': None
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'glm-5.3',
                'system_fingerprint': None,
                'id': '2026111611033559e88cc9fd2240c0',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a1e24f-5f68-7f03-a60c-7f7c4498507b-0',
            tool_calls=[
                {
                    'name': 'search_customer_database',
                    'args': {'query': '李四'},
                    'id': 'call_-7080000459263435397',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 437,
                'output_tokens': 152,
                'total_tokens': 589,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {'reasoning': 121}
            }
        ),
        ToolMessage(
            content='客户记录: 李四, 普通客户, 最近购买日期: 2025-12-20, 累计消费: $3,200',
            name='search_customer_database',
            id='0e2976c9-3539-4285-abaa-5434c73374ad',
            tool_call_id='call_-7080000459263435397'
        ),
        AIMessage(
            content='已找到李四的客户记录。由于李四是**普通客户**而非VIP客户,按照流程不需要发送感谢邮件。现在为您生
成结构化分析报告:',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 230,
                    'prompt_tokens': 506,
                    'total_tokens': 736,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 155,
                        'rejected_prediction_tokens': None
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'glm-5.3',
                'system_fingerprint': None,
                'id': '20261116110339a4ab4e8e423b4441',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a1e24f-6f97-7c40-bb16-07ce9384ff3e-0',
            tool_calls=[
                {
                    'name': 'CustomerAnalysis',
                    'args': {
                        'customer_name': '李四',
                        'customer_tier': '普通客户',
                        'send_email': False,
                        'spending_level': '中'
                    },
                    'id': 'call_-7079995648900068402',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 506,
                'output_tokens': 230,
                'total_tokens': 736,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {'reasoning': 155}
            }
        ),
        ToolMessage(
      

### 1.3 JsonSchema 类型

举例1:

In [23]:
from typing import TypedDict, Annotated

json_schema = {
  "title": "ContactInfo",
  "description": "用户的联系方式",
  "type": "object",
  "properties": {
    "name": {
      "description": "用户姓名",
      "type": "string"
    },
    "email": {
      "description": "用户邮箱地址",
      "type": "string"
    },
    "phone": {
      "description": "用户的手机号",
      "type": "string"
    }
  },
  "required": [
    "name",
    "email",
    "phone"
  ]
}

agent = create_agent(
    model=model,
    response_format=AutoStrategy(schema=json_schema)
)

response = agent.invoke({
    "messages": [
        HumanMessage(content="从这段话中抽取结构化信息: 小明的邮箱是shkstart@atguigu.com, 电话是: 123123123123")
    ]
})

rprint(response)

{
    'messages': [
        HumanMessage(
            content='从这段话中抽取结构化信息: 小明的邮箱是shkstart@atguigu.com, 电话是: 123123123123',
            additional_kwargs={},
            response_metadata={},
            id='a201c151-8dd8-4a46-8428-943b1f0c12d5'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 130,
                    'prompt_tokens': 228,
                    'total_tokens': 358,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 96,
                        'rejected_prediction_tokens': None
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'glm-5.3',
                'system_fingerprint': None,
                'id': '202611161112085fbca01a85ce4f87',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a1e257-3293-77a3-83c0-d8b82cea27a8-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'email': 'shkstart@atguigu.com', 'name': '小明', 'phone': '123123123123'},
                    'id': 'call_-7080000871580299354',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 228,
                'output_tokens': 130,
                'total_tokens': 358,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {'reasoning': 96}
            }
        ),
        ToolMessage(
            content="Returning structured response: {'email': 'shkstart@atguigu.com', 'name': '小明', 'phone': 
'123123123123'}",
            name='ContactInfo',
            id='330ac6f2-6fd0-4674-9437-71370f393da1',
            tool_call_id='call_-7080000871580299354'
        )
    ],
    'structured_response': {'email': 'shkstart@atguigu.com', 'name': '小明', 'phone': '123123123123'}
}

举例2

In [25]:
from typing import Literal, Optional

# 定义工具
@tool(parse_docstring=True)
def search_customer_database(query: str) -> str:
    """
    在客户数据库中搜索信息

    Args:
        query (str): 客户查询字符串, 例如 "张三" 或 "李四"

    Returns:
        str: 客户记录字符串, 包含客户姓名, 等级, 最近购买日期和累计消费
    """
    # 模拟数据库查询结果
    if "张三" in query.lower():
        return "客户记录: 张三, VIP 客户, 最近购买日期: 2026-01-15, 累计消费: $15,000"
    elif "李四" in query.lower():
        return "客户记录: 李四, 普通客户, 最近购买日期: 2025-12-20, 累计消费: $3,200"
    else:
        return f"关于客户{query}, 无记录"

@tool(parse_docstring=True)
def send_email(customer: str) -> str:
    """
    发送感谢邮件

    Args:
        customer (str): 客户名称, 例如 "张三" 或 "李四"

    Returns:
        str: 确认消息, 包含已发送的客户名称
    """
    return f"已向 {customer} 发送感谢邮箱 "

customer_analysis_schema = {
    "title": "CustomerAnalysis",
    "type": "object",
    "description": "客户分析报告",
    "properties": {
        "customer_name": {
            "type": "string",
            "default": "",
            "description": "客户姓名"
        },
        "customer_tier": {
            "type": "string",
            "enum": ["潜在客户", "普通客户", "VIP客户", "流失风险"],
            "default": "潜在客户",
            "description": "客户等级"
        },
        "recent_activity": {
            "type": "string",
            "default": "",
            "description": "最近活动"
        },
        "spending_level": {
            "type": "string",
            "enum": ["低", "中", "高"],
            "default": "低",
            "description": "消费水平"
        },
        "send_email": {
            "type": "boolean",
            "default": False,
            "description": "是否已发送感谢邮件"
        }
    },
    # 所有字段都是必须输出的
    "required": ["customer_name", "customer_tier", "recent_activity", "spending_level"]
}


# 创建智能体
agent = create_agent(
    model=model,
    system_prompt=SystemMessage(content=""
                                "请分析指定客户的情况:"
                                "1. 先搜索客户数据库了解最新情况"
                                "2. 如果是 VIP 客户, 则发送感谢邮件"
                                "3. 基于搜索结果生成结构化分析报告"
                                "4. 如果用户提问与客户记录无关或找不到客户信息, 则返回空对象, 不发送感谢邮件"
                                ),
    tools=[search_customer_database, send_email],
    response_format=ToolStrategy(schema=customer_analysis_schema)
)

# 执行分析
result = agent.invoke({
    # "messages": [{"role": "user", "content": "请分析客户张三"}]
    "messages": [{"role": "user", "content": "请分析客户李四"}]
    # "messages": [{"role": "user", "content": "请分析客户王五"}]
    # "messages": [{"role": "user", "content": "今天天气如何"}]
})

# 处理结果
rprint(result)

{
    'messages': [
        HumanMessage(
            content='请分析客户李四',
            additional_kwargs={},
            response_metadata={},
            id='1c644e4f-2a75-403e-ac83-84b8c3863d65'
        ),
        AIMessage(
            content='我来帮您分析客户李四的情况。首先搜索客户数据库了解最新情况。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 174,
                    'prompt_tokens': 500,
                    'total_tokens': 674,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 143,
                        'rejected_prediction_tokens': None
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'glm-5.3',
                'system_fingerprint': None,
                'id': '20261116111232ac85fd16e6064af9',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a1e257-9142-7e23-b4ff-f67e9a337f94-0',
            tool_calls=[
                {
                    'name': 'search_customer_database',
                    'args': {'query': '李四'},
                    'id': 'call_-7080002383408791474',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 500,
                'output_tokens': 174,
                'total_tokens': 674,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {'reasoning': 143}
            }
        ),
        ToolMessage(
            content='客户记录: 李四, 普通客户, 最近购买日期: 2025-12-20, 累计消费: $3,200',
            name='search_customer_database',
            id='61879fa2-95b0-4242-b178-e549cbe95b5a',
            tool_call_id='call_-7080002383408791474'
        ),
        AIMessage(
            content='已找到李四的客户记录。由于李四是**普通客户**(非VIP客户),根据流程不需要发送感谢邮件。现在为您生
成结构化分析报告:',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 383,
                    'prompt_tokens': 569,
                    'total_tokens': 952,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 291,
                        'rejected_prediction_tokens': None
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'glm-5.3',
                'system_fingerprint': None,
                'id': '20261116111236b2b8a4b401d44abc',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a1e257-a26a-77a1-a9c3-c2ce611a5448-0',
            tool_calls=[
                {
                    'name': 'CustomerAnalysis',
                    'args': {
                        'customer_name': '李四',
                        'customer_tier': '普通客户',
                        'recent_activity': '最近购买日期: 2025-12-20',
                        'send_email': False,
                        'spending_level': '中'
                    },
                    'id': 'call_-7080001730573757368',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 569,
                'output_tokens': 383,
                'total_tokens': 952,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {'rea

### 1.4 @dataclass

举例1

In [26]:
from dataclasses import dataclass

# 2. 使用 Pydantic 结构化方式定义
@dataclass
class ContractInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户的名称")
    email: str = Field(description="用户的邮箱")
    phone: str = Field(description="用户的电话")

agent = create_agent(
    model=model,
    response_format=AutoStrategy(ContractInfo)
)

response = agent.invoke({
    "messages": [
        HumanMessage(content="从这段话中抽取结构化信息: 小明的邮箱是shkstart@atguigu.com, 电话是: 123123123123")
    ]
})

rprint(response)

{
    'messages': [
        HumanMessage(
            content='从这段话中抽取结构化信息: 小明的邮箱是shkstart@atguigu.com, 电话是: 123123123123',
            additional_kwargs={},
            response_metadata={},
            id='5d0247ef-6f27-4d2e-9ef8-3756bca5c66d'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 113,
                    'prompt_tokens': 225,
                    'total_tokens': 338,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 79,
                        'rejected_prediction_tokens': None
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'glm-5.3',
                'system_fingerprint': None,
                'id': '20261116111530abcaf0f876bf46b6',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a1e25a-4765-7a82-99a3-be6698fa066d-0',
            tool_calls=[
                {
                    'name': 'ContractInfo',
                    'args': {'email': 'shkstart@atguigu.com', 'name': '小明', 'phone': '123123123123'},
                    'id': 'call_-7079995992497447526',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 225,
                'output_tokens': 113,
                'total_tokens': 338,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {'reasoning': 79}
            }
        ),
        ToolMessage(
            content="Returning structured response: name='小明' email='shkstart@atguigu.com' phone='123123123123'",
            name='ContractInfo',
            id='9db090cd-a6fa-4ee3-a477-4dbaeeed0e15',
            tool_call_id='call_-7079995992497447526'
        )
    ],
    'structured_response': ContractInfo(name='小明', email='shkstart@atguigu.com', phone='123123123123')
}

举例2

In [27]:
from typing import Literal

# 定义工具
@tool(parse_docstring=True)
def search_customer_database(query: str) -> str:
    """
    在客户数据库中搜索信息

    Args:
        query (str): 客户查询字符串, 例如 "张三" 或 "李四"

    Returns:
        str: 客户记录字符串, 包含客户姓名, 等级, 最近购买日期和累计消费
    """
    # 模拟数据库查询结果
    if "张三" in query.lower():
        return "客户记录: 张三, VIP 客户, 最近购买日期: 2026-01-15, 累计消费: $15,000"
    elif "李四" in query.lower():
        return "客户记录: 李四, 普通客户, 最近购买日期: 2025-12-20, 累计消费: $3,200"
    else:
        return f"关于客户{query}, 无记录"

@tool(parse_docstring=True)
def send_email(customer: str) -> str:
    """
    发送感谢邮件

    Args:
        customer (str): 客户名称, 例如 "张三" 或 "李四"

    Returns:
        str: 确认消息, 包含已发送的客户名称
    """
    return f"已向 {customer} 发送感谢邮箱"

# 定义 Pydantic Schema
@dataclass
class CustomerAnalysis(BaseModel):
    """客户分析报告"""
    customer_name: str = Field(None, description="客户姓名")
    customer_tier: Literal["潜在客户", "普通客户", "VIP客户", "流式风险"] = Field("潜在客户", description="客户等级, 只能是潜在客户, 普通客户, VIP客户或流式风险")
    spending_level: Literal["低", "中", "高"] = Field(None, description="消费水平")
    send_email: bool = Field(False, description="是否已发送感谢邮件")

# 创建智能体
agent = create_agent(
    model=model,
    system_prompt=SystemMessage(content=""
                                "请分析指定客户的情况:"
                                "1. 先搜索客户数据库了解最新情况"
                                "2. 如果是 VIP 客户, 则发送感谢邮件"
                                "3. 基于搜索结果生成结构化分析报告"
                                "4. 如果用户提问与客户记录无关或找不到客户信息, 则返回空对象, 不发送感谢邮件"
                                ),
    tools=[search_customer_database, send_email],
    response_format=ToolStrategy(schema=CustomerAnalysis)
)

# 执行分析
result = agent.invoke({
    # "messages": [{"role": "user", "content": "请分析客户张三"}]
    "messages": [{"role": "user", "content": "请分析客户李四"}]
    # "messages": [{"role": "user", "content": "请分析客户王五"}]
    # "messages": [{"role": "user", "content": "今天天气如何"}]
})

# 处理结果
rprint(result)

{
    'messages': [
        HumanMessage(
            content='请分析客户李四',
            additional_kwargs={},
            response_metadata={},
            id='7ff2e447-54e5-4dd6-a16b-d0c441d43620'
        ),
        AIMessage(
            content='我来帮您分析客户李四的情况。首先搜索客户数据库了解最新情况。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 155,
                    'prompt_tokens': 467,
                    'total_tokens': 622,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 124,
                        'rejected_prediction_tokens': None
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'glm-5.3',
                'system_fingerprint': None,
                'id': '2026111611161753ae43b297de4212',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a1e25b-019b-7e52-a86e-929f19ae3ca5-0',
            tool_calls=[
                {
                    'name': 'search_customer_database',
                    'args': {'query': '李四'},
                    'id': 'call_-7079994652467652809',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 467,
                'output_tokens': 155,
                'total_tokens': 622,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {'reasoning': 124}
            }
        ),
        ToolMessage(
            content='客户记录: 李四, 普通客户, 最近购买日期: 2025-12-20, 累计消费: $3,200',
            name='search_customer_database',
            id='e0296da1-1d40-4d62-82fa-3ad5683aa85f',
            tool_call_id='call_-7079994652467652809'
        ),
        AIMessage(
            content='已找到客户李四的信息。李四是**普通客户**,不符合发送感谢邮件的条件(VIP客户专属),因此不发送感谢
邮件。现在基于搜索结果生成结构化分析报告。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 234,
                    'prompt_tokens': 536,
                    'total_tokens': 770,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 154,
                        'rejected_prediction_tokens': None
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'glm-5.3',
                'system_fingerprint': None,
                'id': '20261116111621dfccb89a27b1477d',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a1e25b-11c4-7d91-8a16-bbf5beeee938-0',
            tool_calls=[
                {
                    'name': 'CustomerAnalysis',
                    'args': {
                        'customer_name': '李四',
                        'customer_tier': '普通客户',
                        'send_email': False,
                        'spending_level': '中'
                    },
                    'id': 'call_-7080003448560679943',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 536,
                'output_tokens': 234,
                'total_tokens': 770,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {'reasoning': 154}
            }
        ),
        ToolMe

### 1.5 多 schema 联合模式

ToolStrategy 允许指定多个类型 Union[类型1, 类型2] 这种写法, LLM 能够根据输入文本的内容, 智能地选择最合适的一个数据模型(Schema)来生成结构化输出, 但是最终会只有一种类型输出.
适用于根本不同输入内容, 生成不同的结构化输出的场景, 但是底层工具转换结构化输出只会转换成一种结构化类型输出

In [29]:
from typing import Union


class ContractInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户的名称")
    email: str = Field(description="用户的邮箱")
    phone: str = Field(description="用户的电话")

class EventInfo(BaseModel):
    """事件详情"""
    event_name: str = Field(description="事件名称")
    date: str = Field(description="事件发生日期")

agent = create_agent(
    model=model,
    response_format=AutoStrategy(Union[ContractInfo, EventInfo])
)

# response = agent.invoke({
#     "messages": [
#         HumanMessage(content="从这段话中抽取结构化信息: 小明的邮箱是shkstart@atguigu.com, 电话是: 123123123123")
#     ]
# })
response = agent.invoke({
    "messages": [
        HumanMessage("从这段话中抽取结构化信息: 2026年高考报名人数突破1200万")
    ]
})

rprint(response)

{
    'messages': [
        HumanMessage(
            content='从这段话中抽取结构化信息: 2026年高考报名人数突破1200万',
            additional_kwargs={},
            response_metadata={},
            id='efe817fb-4daa-4415-bf73-969bfc9450ce'
        ),
        AIMessage(
            content='从这段话中可以抽取到事件信息:',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 206,
                    'prompt_tokens': 286,
                    'total_tokens': 492,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 169,
                        'rejected_prediction_tokens': None
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'glm-5.3',
                'system_fingerprint': None,
                'id': '20261116113433f6c4b9ff00b14efe',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a1e26b-ba43-72d0-a5d9-2def7bf2f24e-0',
            tool_calls=[
                {
                    'name': 'EventInfo',
                    'args': {'date': '2026年', 'event_name': '高考报名人数突破1200万'},
                    'id': 'call_-7080001799293233520',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 286,
                'output_tokens': 206,
                'total_tokens': 492,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {'reasoning': 169}
            }
        ),
        ToolMessage(
            content="Returning structured response: event_name='高考报名人数突破1200万' date='2026年'",
            name='EventInfo',
            id='01800071-4bf3-44cd-b559-4c554bfa922c',
            tool_call_id='call_-7080001799293233520'
        )
    ],
    'structured_response': EventInfo(event_name='高考报名人数突破1200万', date='2026年')
}

### 2. 自定义工具消息: tool_message_content 参数

In [33]:
# 2. 使用 Pydantic 结构化方式定义
class ContractInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户的名称")
    email: str = Field(description="用户的邮箱")
    phone: str = Field(description="用户的电话")

agent = create_agent(
    model=model,
    response_format=ToolStrategy(ContractInfo)
)

response = agent.invoke({
    "messages": [
        HumanMessage(content="从这段话中抽取结构化信息: 小明的邮箱是shkstart@atguigu.com, 电话是: 123123123123")
    ]
})

rprint(response) 

{
    'messages': [
        HumanMessage(
            content='从这段话中抽取结构化信息: 小明的邮箱是shkstart@atguigu.com, 电话是: 123123123123',
            additional_kwargs={},
            response_metadata={},
            id='ce3f6e0b-bbed-49e9-bad9-0f27478025e3'
        ),
        AIMessage(
            content='我从这段话中抽取到的结构化信息如下:',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 157,
                    'prompt_tokens': 225,
                    'total_tokens': 382,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 112,
                        'rejected_prediction_tokens': None
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'glm-5.3',
                'system_fingerprint': None,
                'id': '202611161332313b05a69f33a2402b',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a1e2d7-baca-75e0-9ebe-9bc8ba7454d7-0',
            tool_calls=[
                {
                    'name': 'ContractInfo',
                    'args': {'email': 'shkstart@atguigu.com', 'name': '小明', 'phone': '123123123123'},
                    'id': 'call_-7079996267375356632',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 225,
                'output_tokens': 157,
                'total_tokens': 382,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {'reasoning': 112}
            }
        ),
        ToolMessage(
            content="Returning structured response: name='小明' email='shkstart@atguigu.com' phone='123123123123'",
            name='ContractInfo',
            id='9448aa59-39f0-42cd-9b33-d63286bda12c',
            tool_call_id='call_-7079996267375356632'
        )
    ],
    'structured_response': ContractInfo(name='小明', email='shkstart@atguigu.com', phone='123123123123')
}

作为对比

In [34]:
# 2. 使用 Pydantic 结构化方式定义
class ContractInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户的名称")
    email: str = Field(description="用户的邮箱")
    phone: str = Field(description="用户的电话")

agent = create_agent(
    model=model,
    response_format=ToolStrategy(
        schema = ContractInfo,
        tool_message_content="已成功抽取信息"
    )
)

response = agent.invoke({
    "messages": [
        HumanMessage(content="从这段话中抽取结构化信息: 小明的邮箱是shkstart@atguigu.com, 电话是: 123123123123")
    ]
})

rprint(response) 

{
    'messages': [
        HumanMessage(
            content='从这段话中抽取结构化信息: 小明的邮箱是shkstart@atguigu.com, 电话是: 123123123123',
            additional_kwargs={},
            response_metadata={},
            id='ffb6fa07-b949-4c0b-b578-ef8517aff7c7'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 170,
                    'prompt_tokens': 225,
                    'total_tokens': 395,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 136,
                        'rejected_prediction_tokens': None
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'glm-5.3',
                'system_fingerprint': None,
                'id': '20261116133235a7e0746877b24691',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a1e2d7-cc82-7ec1-822a-d7876a988acd-0',
            tool_calls=[
                {
                    'name': 'ContractInfo',
                    'args': {'email': 'shkstart@atguigu.com', 'name': '小明', 'phone': '123123123123'},
                    'id': 'call_-7080001627494544853',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 225,
                'output_tokens': 170,
                'total_tokens': 395,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {'reasoning': 136}
            }
        ),
        ToolMessage(
            content='已成功抽取信息',
            name='ContractInfo',
            id='1ce980f2-da5b-4fd4-9606-21e4440f01b5',
            tool_call_id='call_-7080001627494544853'
        )
    ],
    'structured_response': ContractInfo(name='小明', email='shkstart@atguigu.com', phone='123123123123')
}